# 02 — Feature Engineering

Malicious URL Detection — Final Year Project

This notebook walks through the 25 lexical URL features implemented in
`src/feature_engineering.py`, demonstrates each extractor individually on
illustrative examples, shows how they combine into `extract_features`, and
verifies robust handling of malformed input.

**Constraint reminder:** every feature here is derived purely from the URL
string — no WHOIS, DNS, or external API calls are made anywhere in this
pipeline, consistent with the project's lexical-only feature specification.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import Image, display

from src import feature_engineering as fe
from src.preprocessing import load_raw_dataset, clean_and_encode

pd.set_option("display.max_columns", 30)

## The 25 features at a glance

| Feature | What it captures |
|---|---|
| `url_length` | Total character length of the URL |
| `domain_length` | Character length of the registered domain |
| `num_dots` | Count of `.` — high counts can indicate subdomain abuse |
| `num_digits` | Count of numeric digits |
| `num_hyphens` | Count of `-` — common in typosquatting domains |
| `num_slashes` | Count of `/` |
| `num_question_marks` | Count of `?` (query delimiter) |
| `num_equal_signs` | Count of `=` (query parameter assignment) |
| `num_ampersands` | Count of `&` (query parameter separator) |
| `num_underscores` | Count of `_` |
| `num_percent_signs` | Count of `%` — often indicates URL-encoded/obfuscated content |
| `num_at_symbols` | Count of `@` — can obscure the true destination host |
| `num_tildes` | Count of `~` |
| `num_commas` | Count of `,` |
| `num_semicolons` | Count of `;` |
| `num_asterisks` | Count of `*` |
| `num_colons` | Count of `:` |
| `num_double_slashes` | Extra `//` beyond the scheme — a redirect-obfuscation pattern |
| `has_https` | 1 if scheme is https |
| `has_ip_address` | 1 if the host is a literal IP address, bypassing domain reputation |
| `num_subdomains` | Number of subdomain labels |
| `shannon_entropy` | Randomness of the URL string (bits) — high entropy suggests obfuscation/DGA |
| `num_suspicious_keywords` | Count of curated phishing-associated keywords |
| `path_length` | Character length of the URL path |
| `query_length` | Character length of the query string |


## Individual extractors on illustrative examples

In [2]:
examples = {
    "Plain safe URL": "http://example.com",
    "Safe URL with subdomain + path": "https://blog.example.com/2024/article",
    "Raw IP host (suspicious)": "http://192.168.1.1/admin/login.php",
    "Keyword-stuffed phishing-style URL": "http://secure-login-verify.paypal-account-update.tk/confirm?id=1",
    "Obfuscated/encoded-looking URL": "http://ex4mpl3-%73ite91.com/wp-admin//login?token=8f92ac",
}

for label, url in examples.items():
    print(f"--- {label} ---")
    print(f"  url_length            = {fe.url_length(url)}")
    print(f"  domain_length         = {fe.domain_length(url)}")
    print(f"  num_dots              = {fe.num_dots(url)}")
    print(f"  num_hyphens           = {fe.num_hyphens(url)}")
    print(f"  has_https             = {fe.has_https(url)}")
    print(f"  has_ip_address        = {fe.has_ip_address(url)}")
    print(f"  num_subdomains        = {fe.num_subdomains(url)}")
    print(f"  shannon_entropy       = {fe.url_shannon_entropy(url):.3f}")
    print(f"  num_suspicious_keywords = {fe.num_suspicious_keywords(url)}")
    print()

--- Plain safe URL ---
  url_length            = 18
  domain_length         = 11
  num_dots              = 1
  num_hyphens           = 0
  has_https             = 0
  has_ip_address        = 0
  num_subdomains        = 0
  shannon_entropy       = 3.614
  num_suspicious_keywords = 0

--- Safe URL with subdomain + path ---
  url_length            = 37
  domain_length         = 11
  num_dots              = 2
  num_hyphens           = 0
  has_https             = 1
  has_ip_address        = 0
  num_subdomains        = 1
  shannon_entropy       = 4.229
  num_suspicious_keywords = 0

--- Raw IP host (suspicious) ---
  url_length            = 34
  domain_length         = 11
  num_dots              = 4
  num_hyphens           = 0
  has_https             = 0
  has_ip_address        = 1
  num_subdomains        = 0
  shannon_entropy       = 4.006
  num_suspicious_keywords = 2

--- Keyword-stuffed phishing-style URL ---
  url_length            = 64
  domain_length         = 24
  num_dots           

## `extract_features` — the combined, single-URL entry point

This is the exact function reused (unchanged) at both training time
(`build_feature_matrix`, below) and inference time (`src/predict.py`), which
is what eliminates train/serve skew in this project.

In [3]:
rows = []
for label, url in examples.items():
    features = fe.extract_features(url)
    features["example"] = label
    rows.append(features)

examples_df = pd.DataFrame(rows).set_index("example")
examples_df[list(fe.FEATURE_NAMES)]

,url_length,domain_length,num_dots,num_digits,num_hyphens,num_slashes,num_question_marks,num_equal_signs,num_ampersands,num_underscores,num_percent_signs,num_at_symbols,num_tildes,num_commas,num_semicolons,num_asterisks,num_colons,num_double_slashes,has_https,has_ip_address,num_subdomains,shannon_entropy,num_suspicious_keywords,path_length,query_length
example,,,,,,,,,,,,,,,,,,,,,,,,,
Plain safe URL,18,11,1,0,0,2,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,3.614369,0,0,0
Safe URL with subdomain + path,37,11,2,4,0,4,0,0,0,0,0,0,0,0,0,0,1,0,1,0,1,4.229327,0,13,0
Raw IP host (suspicious),34,11,4,8,0,4,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,4.006437,2,16,0
Keyword-stuffed phishing-style URL,64,24,2,1,4,3,1,1,0,0,0,0,0,0,0,0,1,0,0,0,1,4.540874,7,8,4
Obfuscated/encoded-looking URL,56,20,1,9,2,5,1,1,0,0,1,0,0,0,0,0,1,1,0,0,0,4.733445,4,16,12


## Robustness: malformed / edge-case input

`extract_features` degrades a single failing feature to `0` (with a logged
warning) rather than crashing the whole extraction pipeline for one bad row —
important at ~300,000-row scale where a handful of malformed entries are
expected.

In [4]:
edge_cases = [
    "http://???///:::",
    "ftp://weird-scheme.example.com/file",
    "http://" + "a" * 300 + ".com",  # extremely long domain label
    "   http://example.com/  ",       # leading/trailing whitespace
]

for url in edge_cases:
    try:
        result = fe.extract_features(url)
        print(f"OK  | {url[:60]!r:63} | url_length={result['url_length']}, entropy={result['shannon_entropy']:.2f}")
    except Exception as exc:
        print(f"FAIL| {url[:60]!r:63} | {exc}")

# Empty / non-string input correctly raises ValueError rather than silently proceeding
for bad_input in ["", "   ", None]:
    try:
        fe.extract_features(bad_input)
        print(f"UNEXPECTED: no error for {bad_input!r}")
    except ValueError as exc:
        print(f"Correctly raised ValueError for {bad_input!r}: {exc}")

OK  | 'http://???///:::'                                              | url_length=16, entropy=2.35
OK  | 'ftp://weird-scheme.example.com/file'                           | url_length=35, entropy=4.07
OK  | 'http://aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa'  | url_length=311, entropy=0.33
OK  | '   http://example.com/  '                                      | url_length=24, entropy=3.57
Correctly raised ValueError for '': url must be a non-empty string.
Correctly raised ValueError for '   ': url must be a non-empty string.
Correctly raised ValueError for None: url must be a non-empty string.


## Building the full feature matrix (as used in training)

`build_feature_matrix` applies `extract_features` across an entire DataFrame of
URLs and preserves any non-URL columns (e.g. `label`) alongside the 25 feature
columns, in a fixed column order (`FEATURE_NAMES`).

In [5]:
raw_df = load_raw_dataset()
clean_df, _report = clean_and_encode(raw_df)

feature_matrix = fe.build_feature_matrix(clean_df.sample(min(500, len(clean_df)), random_state=42))
print(f"Feature matrix shape: {feature_matrix.shape}")
feature_matrix.head()

2026-08-07 10:15:54 | INFO     | src.preprocessing | Loading raw dataset from /home/claude/malicious_url_detector/data/raw/malicious_urls_dataset.csv


2026-08-07 10:15:54 | INFO     | src.preprocessing | Loaded 2500 raw rows


2026-08-07 10:15:54 | INFO     | src.preprocessing | Cleaning complete: 2500 -> 1647 rows (removed 0 unmapped-label, 0 missing-URL, 853 duplicate). Class balance: {'Malicious': 1000, 'Safe': 647}


2026-08-07 10:15:54 | INFO     | src.feature_engineering | Extracting 25 lexical features for 500 URLs


Extracting features:   0%|          | 0/500 [00:00<?, ?it/s]

Extracting features: 100%|██████████| 500/500 [00:00<00:00, 17539.41it/s]

2026-08-07 10:15:54 | INFO     | src.feature_engineering | Feature matrix built: shape=(500, 26)


Feature matrix shape: (500, 26)


,url_length,domain_length,num_dots,num_digits,num_hyphens,num_slashes,num_question_marks,num_equal_signs,num_ampersands,num_underscores,num_percent_signs,num_at_symbols,num_tildes,num_commas,num_semicolons,num_asterisks,num_colons,num_double_slashes,has_https,has_ip_address,num_subdomains,shannon_entropy,num_suspicious_keywords,path_length,query_length,label
680,32,11,1,4,0,3,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,4.327820,0,7,6,0
1412,31,11,1,3,0,4,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,4.066784,0,13,0,0
724,61,18,3,6,1,3,1,1,0,0,0,0,0,0,0,0,1,0,0,0,1,4.318628,0,10,14,1
1358,64,19,2,10,2,3,1,2,1,0,0,0,0,0,0,0,1,0,0,0,1,4.560329,3,7,19,1
1274,31,11,1,3,0,3,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,4.324849,0,7,5,0


## Feature value ranges

A quick sanity check that no feature is producing pathological values (e.g.
negative counts, NaNs) before this matrix is handed to `train.py`.

In [6]:
feature_matrix[list(fe.FEATURE_NAMES)].describe().T[["min", "max", "mean", "std"]]

,min,max,mean,std
url_length,16.000,66.000000,47.672000,14.732770
domain_length,9.000,19.000000,14.910000,3.852217
num_dots,1.000,4.000000,2.128000,1.018671
num_digits,0.000,14.000000,6.984000,3.805352
num_hyphens,0.000,2.000000,0.820000,0.910636
num_slashes,2.000,4.000000,3.326000,0.477682
num_question_marks,0.000,1.000000,0.664000,0.472812
num_equal_signs,0.000,2.000000,1.004000,0.823006
num_ampersands,0.000,1.000000,0.340000,0.474183
num_underscores,0.000,0.000000,0.000000,0.000000


## Notes

- All 25 extractors are pure functions of the URL string — independently unit
  tested in `tests/test_feature_engineering.py`.
- `tldextract` is configured (see `feature_engineering._TLD_EXTRACTOR`) to use
  its bundled offline public-suffix list, so domain/subdomain parsing never
  performs network I/O.
- Because `extract_features` is shared verbatim between training and the Flask
  app's live inference path (`src/predict.py`), any change to a feature
  extractor automatically applies consistently everywhere — there is no
  separate "serving" implementation to keep in sync.

Next: `03_ModelComparison.ipynb` trains all five candidate models on this
feature matrix, tunes three of them, and selects the best one.